In [0]:
%sql
CREATE CATALOG IF NOT EXISTS we47catalog1;
CREATE SCHEMA IF NOT EXISTS we47catalog1.we47schema;
CREATE VOLUME IF NOT EXISTS we47catalog1.we47schema.we47_volume;
     

In [0]:
dbutils.fs.mkdirs("/Volumes/we47catalog/we47schema/we47_volume/we47_dir1")

In [0]:
sampledata="sample data to display"
dbutils.fs.put("/Volumes/we47catalog/we47schema/we47_volume/we47_dir1/samplefile.txt",sampledata,True)
dbutils.fs.ls("/Volumes/we47catalog/we47schema/we47_volume/we47_dir1")
print(dbutils.fs.head("dbfs:///Volumes/we47catalog/we47schema/we47_volume/we47_dir1/samplefile.txt"))

In [0]:
from pyspark.sql.session import SparkSession
spark = SparkSession.builder.appName("Spark DataFrames").getOrCreate()

In [0]:
df1=spark.read.csv("dbfs:///Volumes/we47catalog/we47schema/we47_volume/we47_dir1/samplefile.txt")
df1.show(20,False)#this show is a DSL function (works only in apache spark), pass number of rows to display and truncate of data upto 20 chars true/false
     

In [0]:
df1=spark.read.csv("/Volumes/we47catalog/we47schema/we47_volume/we47_dir1/custs")
#df1.show(20,False)
df1.printSchema()
     


In [0]:
#We use toDF if we want to add column name explicitly, if the data doesnt has column names or if we want to change the column name the data is provided with
df1=spark.read.csv(path="/Volumes/we47catalog/we47schema/we47_volume/we47_dir1/custs").toDF("id","fname","lname","age","profession")
df1.printSchema()
df1.show(2)
display(df1.limit(10))#Display is a special function in databricks, help us display in a table format with a limit that we can define to control the rows
     

In [0]:
#We use header option to consider the column name from the data itself
df1=spark.read.csv(path="/Volumes/we47catalog/we47schema/we47_volume/we47_dir1/custs_header",header=True)
#We use header option to consider the column name from the data itself and we can override the source given columns to our custom column names also by using header+ toDF()...
print(df1.count())
df1.printSchema()
df1.show(2)
df1=spark.read.csv(path="/Volumes/we47catalog/we47schema/we47_volume/we47_dir1/custs_header",header=True).toDF("id","fname","lname","age","occupation")
print(df1.count())
df1.printSchema()
df1.show(2)
#display(df1.limit(10))#Display is a special function in databricks, help us display in a table format with a limit that we can define to control the rows
     

In [0]:
df1=spark.read.csv(path="/Volumes/we47catalog/we47schema/we47_volume/we47_dir1/custs_header",header=True,inferSchema=True)
df1.printSchema()
     

In [0]:
#4. Using delimiter or seperator option
df1=spark.read.csv(path="/Volumes/we47catalog/we47schema/we47_volume/we47_dir1/custs_header_oth_del",header=True,inferSchema=True,sep='~')

#Do we need to put all these efforts?? No
#df1.createOrReplaceTempView("view1")
#spark.sql("select split(`custid~fname~lname~age~profession`,'~') lst_of_cols from view1").createOrReplaceTempView("view2")
#spark.sql("select lst_of_cols[0] id,lst_of_cols[1] fname,lst_of_cols[2] lname,lst_of_cols[3] age,lst_of_cols[4] profession from view2").show(2)
df1.printSchema()
df1.show()

In [0]:
#5. Using different options to create dataframe with csv and other module... (2 methodologies (spark.read.inbuiltfunction or spark.read.format(anyfunction).load("path")) with 3 ways of creating dataframes (pass parameters to the csv()/option/options))
#Methodology #1 (spark.read.inbuiltfunction) we use to create dataframe from builtin sources
df1=spark.read.csv(path="/Volumes/we47catalog/we47schema/we47_volume/we47_dir1/custs_header_oth_del",header=True,inferSchema=True,sep='~')
print(df1.count())
#Methodology #2 (spark.read.format('anysources/anyformats').load()) we use to create dataframe from builtin sources
df1_bq=spark.read.format("csv").load("/Volumes/we47catalog/we47schema/we47_volume/we47_dir1/custs_header")
print(df1_bq.count())

#Mostly we use option or options for external sources, not for builtin sources...
#option can be used for 1 or 2 option...
df1=spark.read.option("header","true").option("sep","~").csv(path="/Volumes/we47catalog/we47schema/we47_volume/we47_dir1/custs_header_oth_del")
#or
df1_bq=spark.read.option("header","true").format("csv").load("/Volumes/we47catalog/we47schema/we47_volume/we47_dir1/custs_header")
print(df1_bq.count())
#options can be used for multiple options in one function as a parameter...
df1=spark.read.options(header=True,sep="~",inferSchema=True).csv(path="/Volumes/we47catalog/we47schema/we47_volume/we47_dir1/custs_header_oth_del")
print(df1.count())
#or
df1_bq=spark.read.options(header=True,sep="~",inferSchema=True).format("csv").load("/Volumes/we47catalog/we47schema/we47_volume/we47_dir1/custs_header")
print(df1_bq.count())

In [0]:
#Methodology #1 (spark.read.inbuiltfunction) we use to create dataframe from builtin sources
df1=spark.read.csv(path="/Volumes/we47catalog/we47schema/we47_volume/we47_dir1/custs_header_oth_del",header=True,inferSchema=True,sep='~')
print(df1.count())

In [0]:
#options can be used for multiple options in one function as a parameter...
#Methodology #2 (spark.read.format('anysources/anyformats').load()) we use to create dataframe from builtin sources
df1_bq=spark.read.option("header","true").option("inferSchema","true").format("csv").load("/Volumes/we47catalog/we47schema/we47_volume/we47_dir1/custs_header")
df1_bq=spark.read.options(header=True,sep="~",inferSchema=True).format("csv").load("/Volumes/we47catalog/we47schema/we47_volume/we47_dir1/custs_header")
print(df1_bq.count())

In [0]:
df1_multiple_files=spark.read.csv(path="/Volumes/workspace/default/volumewe47_datalake/we47_source/custs*",inferSchema=True).toDF("id","fname","lname","age","prof")
print(df1_multiple_files.count())
     

In [0]:
#What if we have files in multiple names in multiple paths... Below code can read data from multiple paths and multiple files...
#Caveate: while reading from multiple csv files.. data patterns like type of column and number of columns should always be same
df1_multiple_files=spark.read.csv(path=["/Volumes/workspace/default/volumewe47_datalake/we47_source/custs*","/Volumes/workspace/default/volumewe47_datalake/we47_source/prospect*","/Volumes/workspace/default/volumewe47_datalake/wd36_source/"],inferSchema=True).toDF("id","fname","lname","age","prof")
print(df1_multiple_files.count())

In [0]:
#recursiveFileLookup is a parameter that can be used to read data from multiple subdirectories
#pathGlobFilter is the parameter used for matching a given pattern under the main/subdirectories
df1_different_subdirs=spark.read.csv("/Volumes/workspace/default/volumewe47_datalake/wd36_source/",inferSchema=True,recursiveFileLookup=True,pathGlobFilter="custs*")
df1_different_subdirs.count()

In [0]:
###Option 1. Using simple string format of define schema. (least used because it will not support some complex formats)
datastruct="id int,fname string,lname string, age int,prof string"
df1=spark.read.schema(datastruct).csv(path="/Volumes/workspace/default/volumewe47_datalake/we47_source/custs1")#we didnt use inferSchema or .toDF
df1.printSchema()
df1.show(3)

In [0]:
#IMPORTANT: 2. Using structure type to define schema.
#For the given data "id,fname,lname,age,prof" i am going to use StructType(StructField("id",IntegerType()),StructField("fname",StringType())...)
#Pattern: StructType([StructField("colname",DataType()),StructField("colname",DataType())......])
#StructType - 1 time
#StructField - total number of fields time
#IntegerType - 2 times
#StringType - 3 times
#4000001|Kristina|Chung|55|Pilot|
from pyspark.sql.types import StructType,StructField,IntegerType,StringType
custom_schema=StructType([StructField("id",IntegerType(),False),StructField("fname",StringType()),StructField("lname",StringType()),StructField("age",IntegerType()),StructField("profession",StringType())])
df1=spark.read.schema(custom_schema).csv(path="/Volumes/workspace/default/volumewe47_datalake/we47_source/custs1")#we didnt use inferSchema or .toDF
df1.printSchema()
df1.show(3)
     

In [0]:
df1=spark.read.csv("/Volumes/workspace/default/volumewe47_datalake/serialized_compressed_data_sources/csv_targetdata",header=True,sep='~')
display(df1.limit(10))
     

In [0]:
df_json=spark.read.json("/Volumes/workspace/default/volumewe47_datalake/serialized_compressed_data_sources/json_targetdata")
display(df_json.take(10))
     

In [0]:
df_xml=spark.read.xml("/Volumes/workspace/default/volumewe47_datalake/serialized_compressed_data_sources/xml_targetdata",rowTag="customer")
display(df_xml.take(10))
     

In [0]:
df_orc=spark.read.orc("/Volumes/workspace/default/volumewe47_datalake/serialized_compressed_data_sources/orc_targetdata")
display(df_orc.take(10))
     

In [0]:
df_parquet=spark.read.parquet("/Volumes/workspace/default/volumewe47_datalake/serialized_compressed_data_sources/parquet_targetdata")#DSL FBP methodology
display(df_parquet.take(10))

In [0]:
df_delta_parquet=spark.read.format("delta").load("/Volumes/workspace/default/volumewe47_datalake/serialized_compressed_data_sources/delta_targetdata")
display(df_delta_parquet.take(10))

In [0]:
df_delta_table=spark.read.table("default.cust")
df2=df_delta_table.select("customerid","BranchID","Address","DateOfBirth").filter("branchid=115")#Domain Specific Lang Function based programming
display(df2.take(10))
#or directly write query
df1=spark.sql("select customerid,branchid,address,dateofbirth from default.cust where branchid=115")#SQL Declarative Query
display(df1.take(10))
     